# Benchmark findings

Short statements of what the benchmark studies found about fitted models and
analyst workflows, measured against planted structure, held-out error or
competing libraries. Each result belongs to the data, settings and library
version recorded in its study; the linked study page has the design and the
full tables. Where a study measured a configuration other than the default, the statement names the configuration measured.

## Capacity selection and transfer

**Fixed and selected capacity.** On the competitive shootout panels (see
[](records/competitive_shootout.md)), an additive
model with fixed capacity was a median 46.5% behind the best competing
library per panel. The usual remedy, an always-on joint tree, turned the
cancer panel from 11% ahead of the best competitor to 109% behind. The
capacity that predicts best differs between data sets.
`select_capacity` scores three nested specifications out of fold on one
shared fold partition: the base formula, the base plus screened tensor
products, and that plus one joint tree. It keeps the smallest specification
whose score is within two standard errors of the best. Run on the same
panels, it reduced the cancer excess from +104.0% to +7.6%, kept the smooth
specification where it was best (`smooth_sparse` -0.9%,
`correlated_proxies` -3.5%, both ahead of the best competitor) and reduced
`concrete` from +19.6% to +12.7% (excess: percent increase in held-out loss
over the best competing library; lower is better). The joint tree reads
factor columns as well as numeric ones, a pair is proposed for a tensor
product only when both columns have more than two distinct values, and each
specification has its own step count chosen out of fold.

**Switching conditional effects in `effect_model`.** The shootout's causal
arm planted a treatment effect that changes sign inside the range of a
modifier (`1 + 1.5 sign(x0 x1)`). With only additive effect dictionaries
the PEHE (root mean squared error of the per-unit effect; lower is better)
was 1.548. `effect_model` therefore has a third candidate, a depth-3 `btree`
over all modifiers fitted with `selection=sse`, adopted only when its
cross-validated R-loss (the loss of the residual-on-residual effect
regression) is at least 5% below that of the best additive candidate. On
seven confirmation draws, disjoint from the three draws that chose the
depth, the switching panel reached PEHE 1.255 and selected the tree in 7 of
7 draws, while the three additive panels selected the same dictionary as
before in all 21 draws. A tree effect gives no one-curve-per-modifier
decomposition, so the fitted effect is harder to read.

**Transfer from CatBoost summaries.** Two probes built an mboostrs
specification from a fitted, tuned CatBoost model: one from its native
feature importances and top interaction pair, the other from spline degrees
of freedom chosen to reproduce its partial-dependence curves to 2% relative
residual. Both had higher test error than the tuned hybrid specification
built without CatBoost (test mean squared error, lower is better: protein
21.2 against 18.9; superconductivity 178 against 140; housing 0.0595 against
0.0555). The probes do not identify which part of the transfer raised the
error. A specification derived from another model's summaries needs the
same held-out evaluation as any other candidate. The records are
`docs/benchmarks/distillation_probe.json` and
`docs/benchmarks/shape_transfer_probe.json`.

## PM2.5 forecasting

The study is [](../benchmarks/pm25_forecast_competition); its artifacts are
under `docs/book/benchmarks/pm25_forecast/`.

**Reference-model comparison.** An additive formula, tuned by an analyst,
reached 2014 RMSE of log PM2.5 of 0.6615, against 0.6181 for CatBoost (7% higher). The tuning did not compare the formula with a flexible reference
model. `reference_gap` measures that comparison, and the agent guide's
`modeling-tactics` topic asks for it. A specification with lagged
weather, tensor interactions and wind-direction `by=` terms, each chosen on
the validation year, reached 2014 RMSE 0.636, which closes 58% of the gap;
on the validation year its median pinball loss was lower than CatBoost's
(21.21 against 21.27).

**Response transformation.** The response is positive, with a 99th
percentile near 400 µg/m³ and a median near 70. Fitted on the raw scale,
the mean pinball loss over three quantile levels was 21.2; fitted to
`log1p` of the response, it was 14.0. Quantiles are preserved by a
monotone transform, so the log-scale quantiles map back exactly.

**Capacity selection on rolling folds.** `select_capacity`, started from
the analyst's submitted formula with rolling-origin folds, scored all three
specifications within noise (0.2511, 0.2507, 0.2518) and kept the base;
its 2014 RMSE was 0.6604 against 0.6615 for the submission. The gains of the
manual search came from new features (lags), not from added capacity.

**Limitations.** The 80% prediction interval covers 0.74 of 2014 hours
overall and 0.68 in winter, for every library. With five rolling folds, the
two-standard-error rule of `select_capacity` requires an improvement of
roughly 14% before it adopts a larger specification, which is large
relative to the differences in this comparison.

## Claim-cost discovery

The study is [Claim-cost discovery](../benchmarks/claims_discovery).

**Failed calls.** Refits, confidence intervals and conformal
calibration accept string category labels, and an empty selection returns the intercept-only model. The agent guide describes the available intervals and includes worked examples. Under the automated-mode scorecard, mboostrs scores 3.8 and statsmodels 3.5.

**Missing covariates.** Housing age was missing on 4-16% of rows, and roof
class was assigned partly from age, so filling the gaps with the median let
the roof-class coefficients absorb the age effect. `na_action="impute_model"`
fills each gap from the other columns with a penalized additive regression
and keeps a missingness indicator; with it, roof-class interval coverage was
0.80, against 0.30 with the median fill. An alternative that split each
smooth on a filled column into observed and filled parts lowered
housing-age coverage from 0.90 to 0.60 and is not in the library.

**Out-of-range covariates.** One test row had an average occupancy of
1,243 persons per household against a training maximum of 6.57, and the
linear extension of the spline beyond the boundary predicted 338 for a true
value near 0. The `extrapolation=` argument and the per-term out-of-range
warnings report such rows.

**Recovery measures as regression checks.** Interaction recall and effect
recall on the planted data detect changes that the unit tests do not: one
change to the screening pipeline lowered interaction recall from 0.81 to
0.38 and effect recall from 0.58 to 0.33 without failing a test.

**Narrow spatial effects.** A spatial bump 0.4 degrees wide is not
resolvable with a 12-by-12 basis, and the 90% intervals of every library
covered it on at most 1 of 8 replicates. A larger basis is needed for
narrow features; `mb.gam` refits fast enough to compare several sizes.

## Causal-effect estimation

The study is [Causal discovery](../benchmarks/causal_discovery).

**Per-arm outcome models in thin overlap.** With separate outcome models
for each arm, the doubly robust (AIPW) estimate extrapolated on rows with
cross-fitted propensities near 0 and 1: the mean absolute error of the
average effect reached \$5.57 million, and the no-effect data drew false
claims. On 12 simulated job-training draws evaluated without an analyst,
`aipw(outcome_model="auto")` had average-effect RMSE \$1,019 and a mean
absolute error of \$28 on the two no-effect draws, and
`effect_model(terms="auto")` had mean top-20% ranking regret 0.34 (0 is
the oracle ranking, 1 a random one). One automatic `effect_model` fit on
10,074 rows took 36 s.

**Analyst runs.** Over four mboostrs analyst runs the weighted scorecard
score ranged from 3.03 to 3.21 in fully automated mode and from 3.06 to
3.26 in semi-automated mode. The fourth run reproduced exactly and had the
best effect-query coverage of all libraries, 0.91. Its top-20% ranking
regret was 0.77, because its effect model used education as the only
modifier.

**Limitations.** statsmodels' weighted score, 3.53, is higher than that of
mboostrs. In the last analyst run the average-effect interval covered the
truth on 5 of 8 replicates, against 7 of 8 in the run before it. The
adjustment-set (causal structure) grade was 2 in every analyst run.

## Discontinuity screening

**Diagonal regime boundaries.** The planted panel has a jump of
+2.0 across the boundary `s1 = 0.45 + 0.35 s0` at n=800 with noise standard
deviation 0.5. Smooth fits absorb most of a step, axis-aligned step
candidates show no cross-validated gain, and the workflow does not
propose diagonal boundaries. An xgboost analysis found the boundary by scanning diagonal indicators on a grid. `screen_discontinuities` scans
axis-aligned and diagonal step boundaries. It ranks candidates on one half
of the rows, estimates the jump and its z statistic on the other half, and
types each top row as a jump or a steep smooth transition by comparing the
level gap measured at two scales (a jump has the same gap at both). On the
planted panel the boundary was recovered with 99.2% row agreement (jump
1.965 of the planted 2.0, held-out z 16.1), and the step indicator of that
boundary reduced out-of-fold MSE by 0.126 relative to the additive baseline
(5.2 paired standard errors), while the `btree(pair)` candidate, the default
when the boundary is unknown, did not improve on the baseline. In 200
replicates without a jump, the top row was flagged on 1.0% of screens, the
mean absolute z was 0.81 against a theoretical 0.80, and steep smooth
crossings were flagged on the raw response but typed as jumps on only 8% of
screens.

## Double-penalty REML

The pilot asked whether an additional penalty on the null space of each
smoother (the double penalty, which lets REML shrink a whole term to zero)
can separate dropping a term from smoothing it. It used the REML engine on
four Gaussian scenarios with 20 seeds each. The result depended on the
scenario: the double penalty reduced signal MSE (error against the
noiseless mean function) by 26% on sparse signals and by 57% with a
correlated substitute, but raised it by 30% on dense signals, and it failed
the across-scenario criterion fixed before the runs, so it is not the
default. The recommendation is to choose among boosting, joint REML and
double-penalty REML on common inner folds of the training data. In the same
table, joint REML over all terms had lower error than tuned boosting in
every scenario with signal at n=600, p=8, and took less time than
boosting's tuning; boosting had the lowest error in the null scenario.